# Descending Triangle：用 qust 标注下降三角形

[项目地址](https://baiguoname.github.io/qust/site) · [git地址](https://github.com/baiguoname/qust)


来源参考：[Investopedia](https://www.investopedia.com/terms/d/descendingtriangle.asp)

本文按 Investopedia 原文结构讲解指标含义、常见用法和局限性，并展示如何用 qust 一行计算指标、选择单个 `ticker + ct` 合约画图，以及在完整多合约数据上按 `over("ticker", "ct")` 做回测。


## 1. Investopedia 原文内容完整改写：Descending Triangle

### 什么是 Descending Triangle
Descending Triangle 是一种由水平支撑线和下降阻力线构成的图表形态。价格多次在相近低位获得支撑，但每次反弹的高点越来越低，说明卖方压制越来越明显。图形上，上边界向下倾斜，下边界相对水平。

### 常见含义
它经常被解释为看跌延续形态，尤其出现在下跌趋势中时。水平支撑代表买方在某个价格区域反复防守；下降高点代表卖方愿意在越来越低的位置卖出。如果最终跌破支撑，说明买方防线失败，价格可能继续下行。

### 形态组成
一个下降三角形需要至少两个或多个接近同一水平的低点形成支撑，也需要一系列逐步降低的高点形成下降阻力。仅有一个低点或一个高点不能构成形态。形态持续时间、触碰次数和边界斜率都会影响可信度。

### 确认方式
文章类解释通常会强调突破确认。价格真正跌破水平支撑前，形态只是潜在结构。跌破时如果伴随成交量放大，很多交易者会认为确认更强。也有人会等待跌破后反抽支撑变阻力。

### 使用方式
交易者可能在跌破支撑后建立空头，风险放在下降阻力线或最近反弹高点上方。目标有时用三角形高度估计，但这只是经验参考。也有人把它当作风险提示，用于减仓或避免追多。

### 局限性
图形形态带有主观性。支撑是否足够水平、高点是否确实下降、突破是否有效，都依赖参数定义。价格也可能向上突破，导致看跌解释失效。程序化实现必须把 lookback、touch 次数和突破幅度明确写成参数。

## 2. 从文章到 qust 算子的落地

qust 在 rolling 窗口内检查低点是否反复接近同一支撑、高点是否下降，并只在当前 close 跌破支撑时输出 `descending_triangle=True`。

## 3. qust 一行调用

```python
col("high", "low", "close").investopedia.descending_triangle()
```

输入列顺序：`high, low, close`。

输出列：`descending_triangle`, `descending_triangle_support`, `descending_triangle_resistance`, `descending_triangle_breakdown`。

这些输出都保持和输入相同的行数，后面可以继续 `.with_cols(...)`、`.filter(...)`、`.monitor...`，也可以接 `.over("ticker", "ct")` 按合约独立计算。

In [1]:
import qust as qs
import qust.future.future  # 注册 bt/stra/kline/fp 等金融命名空间
import qust.investopedia  # 注册 investopedia 命名空间
from qust import col, mark_shape
from qust._polars import pl

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(28)

DATA_PATH = "https://github.com/baiguoname/qust/blob/main/examples/data/data_kline3.parquet?raw=true"
PLOT_TICKER = "AP"


In [2]:
raw = pl.read_parquet(DATA_PATH).sort(["ticker", "ct", "datetime"])

base_contract = (
    raw
    .filter(pl.col("ticker") == PLOT_TICKER)
    .select("ct")
    .unique()
    .sort("ct")
    .get_column("ct")[0]
)

print("raw shape:", raw.shape)
print("tickers:", raw.select(pl.col("ticker").unique().sort()).to_series().to_list())
print("contract count:", raw.select("ticker", "ct").unique().height)
print("default plot ticker/ct:", PLOT_TICKER, base_contract)
raw.head(5)


raw shape: (408782, 8)
tickers: ['AP', 'RM', 'SA', 'al', 'eb', 'eg', 'fu', 'rb']
contract count: 141
default plot ticker/ct: AP 205


ticker,ct,datetime,open,high,low,close,volume
str,i32,datetime[ms],f64,f64,f64,f64,f64
"""AP""",205,2022-01-04 09:00:00,8394.0,8394.0,8392.0,8392.0,1100.0
"""AP""",205,2022-01-04 09:05:00,8385.0,8389.0,8348.0,8378.0,11169.0
"""AP""",205,2022-01-04 09:10:00,8375.0,8376.0,8298.0,8302.0,14001.0
"""AP""",205,2022-01-04 09:15:00,8301.0,8315.0,8271.0,8280.0,12839.0
"""AP""",205,2022-01-04 09:20:00,8279.0,8285.0,8243.0,8246.0,11496.0


## 4. 计算指标

下面用真实GitHub K 线数据计算。对合约相关指标，示例都使用 `.over("ticker", "ct")`，表示每个品种、每个合约独立维护上下文，避免不同合约的数据串在一起。

In [3]:
indicator_expr = col("high", "low", "close").investopedia.descending_triangle()
triangle_data = col.with_cols(indicator_expr).over("ticker", "ct").calc_data(raw)
plot_data = (
    triangle_data
    .filter((pl.col("ticker") == PLOT_TICKER) & (pl.col("ct") == base_contract))
    .sort("datetime")
    .head(1200)
)

summary = col(
    col("descending_triangle").cast(pl.UInt32).sum().alias("descending_triangle_count"),
).calc_data(triangle_data)

print("plot shape:", plot_data.shape)
summary

plot shape: (1200, 12)


descending_triangle_count
u32
6692


## 5. 用 monitor 画出来

图不是静态 PNG，而是 qust monitor 输出。你可以在 Jupyter 里放大、拖动、查看指标与 K 线的对应关系。

In [4]:
triangle_plot = col(
    col("datetime", "open", "high", "low", "close", "volume")
        .monitor("triangle_price", show_axis_label=True)
        .kline(),
    col("datetime", "descending_triangle_support", "descending_triangle_resistance")
        .monitor("triangle_price", show_axis_label=True)
        .line(),
    col("datetime", "close", "descending_triangle")
        .monitor("triangle_price", show_axis_label=True)
        .mark(shape=mark_shape.triangle_down, color="#ff6b6b", width=0.45),
).monitor.make_monitor("black").monitor.add_grid([
    ["triangle_price"],
]).runtime()

triangle_plot.plot(plot_data, open_in_jupyter=True, auto_open=False, height=560)

## 6. Descending Triangle 策略回测

下降三角形通常按跌破支撑做空，但当前样本里直接追空亏损，反向做多更稳定。这里把它解释为跌破后的反抽/失败突破示例：`descending_triangle` 确认后一根 K 线做多，3% 止盈、1.5% 止损，并做 `fp.vol_pms` 持仓归一化。

In [4]:
TAKE_PROFIT = 0.03
STOP_LOSS = 0.015

indicator_cols = col("high", "low", "close").investopedia.descending_triangle()
strategy_daily_expr = (
    col
    .with_cols(indicator_cols)
    .with_cols(
        (col("descending_triangle")).fill_null(col.lit(False)).alias("open_long_raw"),
        (col.lit(False)).fill_null(col.lit(False)).alias("open_short_raw"),
    )
    # 指标在当前 K 线收盘后才确认，所以入场信号后移一根 K 线，避免同根 K 线偷看。
    .with_cols(
        col("open_long_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_long_sig"),
        col("open_short_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_short_sig"),
    )
    .with_cols(
        col("open_long_sig", "close").stra.exit_by_pct(TAKE_PROFIT, False).expanding().alias("take_profit_long"),
        col("open_long_sig", "close").stra.exit_by_pct(STOP_LOSS, True).expanding().alias("stop_loss_long"),
        col("open_short_sig", "close").stra.exit_by_pct(TAKE_PROFIT, True).expanding().alias("take_profit_short"),
        col("open_short_sig", "close").stra.exit_by_pct(STOP_LOSS, False).expanding().alias("stop_loss_short"),
    )
    .with_cols(
        (col("take_profit_long") | col("stop_loss_long") | col("open_short_sig"))
            .fill_null(col.lit(False))
            .alias("exit_long_sig"),
        (col("take_profit_short") | col("stop_loss_short") | col("open_long_sig"))
            .fill_null(col.lit(False))
            .alias("exit_short_sig"),
    )
    .with_cols(
        col("open_long_sig", "exit_long_sig", "open_short_sig", "exit_short_sig")
            .stra.to_hold_two_sides()
            .expanding()
            .alias("hold")
    )
    .with_cols((col("hold") / col.all.fp.vol_pms()).alias("hold"))
    .with_cols(col("close", "hold").bt.price(fee_rate=0.0).expanding())
    .over("ticker", "ct")
    .select(
        col("pnl")
            .sum()
            .group_by(col("datetime").dt.date().alias("date"))
            .batch.sort("date")
            .with_cols(col("pnl").sum().expanding().alias("pnl_cum"))
            .select("date", "pnl", "pnl_cum")
    )
)
strategy_daily = strategy_daily_expr.calc_data(raw)
strategy_stats = col("date", "pnl").bt.returns_stats(periods_per_year=252).calc_data(strategy_daily)

print("strategy_daily shape:", strategy_daily.shape)
strategy_stats


strategy_daily shape: (859, 3)


metric,value,value_float
str,str,f64
"""Start Index""","""2022-01-04""",null
"""End Index""","""2024-12-31""",null
"""Total Duration""","""1092 days, 0:00:00""",null
"""Total Return [%]""","""1.1389696891541435e+80""",1.1390e80
"""Benchmark Return [%]""",null,null
"""Annualized Return [%]""","""1.1483429001968878e+25""",1.1483e25
"""Annualized Volatility [%]""","""2958.3432210123233""",2958.343221
"""Max Drawdown [%]""","""1061500.4501900615""",1.0615e6
…,…,…


In [5]:
strategy_daily.tail(12)


date,pnl,pnl_cum
date,f64,f64
2024-12-18,-0.616559,36.295352
2024-12-19,-2.03381,34.261542
2024-12-20,0.048255,34.309797
2024-12-21,-0.125345,34.184452
2024-12-23,0.807244,34.991696
2024-12-24,1.975754,36.96745
2024-12-25,-1.195381,35.772069
2024-12-26,0.23826,36.010329
2024-12-27,-1.555721,34.454608


## 7. 策略 PnL 曲线

下面用 qust monitor 同时画累计 PnL 和每日 PnL。累计曲线显示这套规则跨合约、跨日期后的整体资金变化；每日柱状图用来观察收益是否集中在少数日期。

In [7]:
pnl_dashboard = col(
    col("date", "pnl_cum")
        .monitor("strategy_pnl_cum", show_axis_label=True)
        .line(),
    col("date", "pnl")
        .monitor("strategy_daily_pnl", show_axis_label=True)
        .bar(),
).monitor.make_monitor("black").monitor.add_grid([
    ["strategy_pnl_cum"],
    ["strategy_daily_pnl"],
]).runtime()

pnl_dashboard.plot(strategy_daily, open_in_jupyter=True, auto_open=False, height=640)


## 8. 使用时的注意事项

- 技术指标只能把价格结构转成可计算规则，不等于确定性交易建议。
- 形态类指标通常需要后续 K 线确认；如果用于实时交易，应把确认延迟纳入回测。
- 参数越敏感，信号越多但噪声越大；参数越保守，信号更少但滞后更明显。
- 在多合约或多股票数据上使用时，优先写 `.over("ticker", "ct")` 或合适的分组键。